# Prospect Client Analysis — Google Colab
This notebook runs the same Google Places → ETL → enrichment → ML → analytics pipeline used by the Streamlit dashboard.


In [ ]:
!git clone https://github.com/softmitend/prospect-client-analysis.git
%cd prospect-client-analysis
!pip -q install -r requirements.txt


In [ ]:
from getpass import getpass
API_KEY = getpass('Google Maps API key: ')
LOCATION = 'Banyuwangi, East Java, Indonesia'
CATEGORIES = ['restaurant', 'cafe', 'hotel', 'dentist', 'clinic', 'beauty salon', 'lawyer', 'car repair']


In [ ]:
from src.pipeline import run_pipeline
businesses, summary = run_pipeline(
    api_key=API_KEY,
    location=LOCATION,
    categories=CATEGORIES,
    max_results_per_category=30,
    enrich_websites=True,
    min_category_size=3,
)
summary


In [ ]:
cols = [c for c in ['business_name','category','website','public_email','whatsapp_number','phone','address','opening_hours_text','digital_presence_score','ml_cluster','ml_anomaly'] if c in businesses]
businesses[cols].head(30)


In [ ]:
import matplotlib.pyplot as plt
plot_df = summary.sort_values('website_adoption_pct')
plt.figure(figsize=(10, max(4, len(plot_df)*0.35)))
plt.barh(plot_df['category'], plot_df['website_adoption_pct'])
plt.xlabel('Website adoption (%)')
plt.title('Website adoption by business category')
plt.tight_layout()
plt.show()


In [ ]:
from src.export import to_excel_bytes
businesses.to_csv('businesses_clean.csv', index=False, encoding='utf-8-sig')
summary.to_csv('category_website_adoption.csv', index=False, encoding='utf-8-sig')
open('prospect_client_analysis.xlsx', 'wb').write(to_excel_bytes(businesses, summary))
print('Exports created.')


## Inspect the ML layer
`ml_cluster` comes from TF-IDF + MiniBatchKMeans over business/category text. `ml_anomaly` comes from Isolation Forest over data-completeness and business-signal features. Neither model is used to fake the website-adoption percentage; that metric remains a deterministic aggregation.
